# Prism Training Race: Time-to-Quality
**Runtime → A100 or T4**

The metric: **how many steps (= compute) to reach a target val loss?**

Three contestants:
1. **Baseline**: Standard Normal(0, 0.02) init
2. **Prism**: Self-extracted spectral init (full 5K steps)
3. **Prism-Taper**: Prism init + aggressive LR decay after step 500

Requires: run `nanogpt_prism_shakespeare.ipynb` first to get the
converged checkpoint and extracted spectra.

In [ ]:
# Cell 1: Setup
import os, subprocess
if os.path.exists('/content/nanogpt-prism'):
    subprocess.run(['rm', '-rf', '/content/nanogpt-prism'])
!git clone https://github.com/realityinspector/nanogpt-prism.git /content/nanogpt-prism
%cd /content/nanogpt-prism
!pip install -q transformers tiktoken datasets
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
!python data/shakespeare_char/prepare.py
print('Ready.')

In [ ]:
# Cell 2: Train teacher model to convergence + extract spectra
# (Skip if already done — checks for cached spectra)
import os, subprocess

if os.path.exists('.prism_cache/shakespeare/spectra.json'):
    print('Spectra already cached. Skipping teacher training.')
else:
    print('Training teacher model (2K steps to best val)...')
    subprocess.run([
        'python', 'train.py', 'config/train_shakespeare_char.py',
        '--max_iters=2000', '--eval_interval=500', '--eval_iters=50',
        '--log_interval=500', '--out_dir=out-teacher',
        '--always_save_checkpoint=True', '--compile=False',
        '--prism_init=False', '--wandb_log=False',
    ], timeout=600)
    print('Extracting spectra...')
    subprocess.run([
        'python', 'prism_extract.py',
        '--ckpt', 'out-teacher/ckpt.pt',
        '--out', '.prism_cache/shakespeare',
    ], timeout=120)
    print('Done.')

In [ ]:
# Cell 3: Run all three contestants
import subprocess, time, re, json

STEPS = 5000
EVAL_INTERVAL = 100  # frequent evals for precise time-to-quality

def run_config(name, extra_args):
    """Run a training config and return parsed results."""
    print(f'\n{"="*60}')
    print(f'  {name}')
    print(f'{"="*60}')
    
    args = [
        'python', 'train.py', 'config/train_shakespeare_char.py',
        f'--max_iters={STEPS}',
        f'--eval_interval={EVAL_INTERVAL}',
        '--eval_iters=50',
        '--log_interval=100',
        f'--out_dir=out-race-{name}',
        '--wandb_log=False',
        '--compile=False',
    ] + extra_args
    
    t0 = time.time()
    result = subprocess.run(args, capture_output=True, text=True, timeout=1200)
    wall_time = time.time() - t0
    
    # Parse val losses
    val_losses = {}
    for line in result.stdout.split('\n'):
        m = re.search(r'step (\d+): train loss ([\d.]+), val loss ([\d.]+)', line)
        if m:
            val_losses[int(m.group(1))] = float(m.group(3))
    
    # Save full log
    log_path = f'/content/race_{name}.txt'
    with open(log_path, 'w') as f:
        f.write(result.stdout)
    
    if result.returncode != 0:
        print(f'ERROR: {result.stderr[-500:]}')
    
    best_val = min(val_losses.values()) if val_losses else 999
    best_step = min(val_losses, key=val_losses.get) if val_losses else 0
    print(f'  Best val: {best_val:.4f} at step {best_step}')
    print(f'  Wall time: {wall_time:.0f}s')
    
    return {'name': name, 'val_losses': val_losses, 'wall_time': wall_time,
            'best_val': best_val, 'best_step': best_step}

results = {}

# 1. Baseline
results['baseline'] = run_config('baseline', [
    '--prism_init=False',
])

# 2. Prism (full run, same LR)
results['prism'] = run_config('prism', [
    '--prism_init=True',
    '--prism_align=0.75',
    '--prism_spectra=.prism_cache/shakespeare/spectra.json',
    '--prism_directions=.prism_cache/shakespeare/directions.pt',
])

# 3. Prism-Taper (Prism init + lower LR + shorter warmup)
# The idea: Prism gives the head start, lower LR prevents overfitting
results['prism_taper'] = run_config('prism_taper', [
    '--prism_init=True',
    '--prism_align=0.75',
    '--prism_spectra=.prism_cache/shakespeare/spectra.json',
    '--prism_directions=.prism_cache/shakespeare/directions.pt',
    '--learning_rate=5e-4',   # half the default 1e-3
    '--warmup_iters=50',      # shorter warmup since already initialized
])

# Save all results
with open('/content/race_results.json', 'w') as f:
    json.dump({k: {kk: vv for kk, vv in v.items() if kk != 'val_losses'}
               for k, v in results.items()}, f, indent=2)
    # val_losses keys are ints, need to convert
with open('/content/race_curves.json', 'w') as f:
    json.dump({k: {str(s): l for s, l in v['val_losses'].items()}
               for k, v in results.items()}, f, indent=2)
print('\nAll results saved.')

In [ ]:
# Cell 4: Time-to-Quality comparison
import json

curves = json.load(open('/content/race_curves.json'))

# Convert keys back to ints
for k in curves:
    curves[k] = {int(s): v for s, v in curves[k].items()}

# Find baseline's best val loss as the quality target
baseline_best = min(curves['baseline'].values())
baseline_best_step = min(curves['baseline'], key=curves['baseline'].get)

print('='*60)
print('  TRAINING RACE: Time to Quality')
print('='*60)
print(f'\nTarget: val_loss ≤ {baseline_best:.4f} (baseline best at step {baseline_best_step})')
print(f'\n{"Runner":>15s}  {"Steps to target":>16s}  {"Speedup":>8s}  {"Best val":>10s}  {"Best step":>10s}')
print('-' * 65)

for name in ['baseline', 'prism', 'prism_taper']:
    c = curves[name]
    best_val = min(c.values())
    best_step = min(c, key=c.get)
    
    # Find first step where val_loss <= target
    target_step = None
    for step in sorted(c.keys()):
        if c[step] <= baseline_best:
            target_step = step
            break
    
    if target_step:
        speedup = f'{baseline_best_step / target_step:.1f}x'
        print(f'{name:>15s}  {target_step:>16d}  {speedup:>8s}  {best_val:>10.4f}  {best_step:>10d}')
    else:
        print(f'{name:>15s}  {"never":>16s}  {"—":>8s}  {best_val:>10.4f}  {best_step:>10d}')

print(f'\n--- Full curves ---')
print(f'{"Step":>6s}', end='')
for name in ['baseline', 'prism', 'prism_taper']:
    print(f'  {name:>12s}', end='')
print()
print('-' * 45)
all_steps = sorted(set().union(*[c.keys() for c in curves.values()]))
for step in all_steps:
    print(f'{step:>6d}', end='')
    for name in ['baseline', 'prism', 'prism_taper']:
        v = curves[name].get(step)
        if v:
            best_at_step = min(curves[n].get(step, 999) for n in curves)
            marker = ' *' if v == best_at_step else '  '
            print(f'  {v:>10.4f}{marker}', end='')
        else:
            print(f'  {"—":>12s}', end='')
    print()

In [ ]:
# Cell 5: Download
from google.colab import files
for f in ['/content/race_results.json', '/content/race_curves.json',
          '/content/race_baseline.txt', '/content/race_prism.txt',
          '/content/race_prism_taper.txt']:
    try: files.download(f)
    except: pass